# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Pollux Gronier  Student ID: B00822392
2. Full name: Saty Viard Laroque  Student ID: B00822312
3. Full name: Fabrice Afopa  Student ID: B00817356
4. Full name: Eliott Beghin  Student ID: B00824999

**Group / repo name:** `aidams-lab4-gronier-viardlaroque-afopa-beghin`
**Submitter (one person):** Pollux Gronier
**Repo URL:** https://github.com/Polluxgnr/Lab_4

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


## Setup (given)

In [1]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA = Path("data") if Path("data").exists() else Path(".")
excerpts = pd.DataFrame(json.loads((DATA / "excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads((DATA / "gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads((DATA / "recorded_responses.json").read_text()))
corpus = json.loads((DATA / "aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [2]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [3]:
# Print excerpts E03, E09, E19 and compare with gold standard
for eid in ["E03", "E09", "E19"]:
    text = excerpts.loc[excerpts["id"] == eid, "text"].values[0]
    print(f"=== Excerpt {eid} ===")
    print(text.strip())
    print()

print("=== Gold standard labels for E03, E09, E19 ===")
display(gold.loc[["E03", "E09", "E19"], FIELDS])


=== Excerpt E03 ===
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

=== Excerpt E09 ===
Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050.

=== Excerpt E19 ===
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.

=== Gold standard labels for E03, E

,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year
id,,,,,,
E03,Lianhe Iron & Steel Group,2023,"41,080,000.000","5,120,000.000",21.400,2060
E09,Ostend Metals,2024,NaN,NaN,4.400,2050
E19,Bluewater Iron,2024,"5,100,000.000","410,000.000",2.990,2050


**Your answer (Q1):**

Hand-labeling E03, E09, and E19 before checking gold exposed three tricky fields: E03's Scope 1 and 2 (reporting unit is "10,000 tonnes", requiring a 10,000x multiplier), E09's Scope 1 (only a combined 8.7 Mt figure is reported, which prompt rules say must be recorded as null), and E19's production (given as 3.3 million short tons, which converts to 2.99 metric tonnes). Within our group, we initially disagreed on E09—one of us entered 8.7 Mt while another left it blank—and on E19, where two of us missed the short-ton conversion. This shows that "gold labels" are not raw empirical measurements but an agreed rulebook of conventions; disagreement between annotators highlights source ambiguity rather than simple carelessness.


## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [4]:
class SteelDisclosure(BaseModel):
    company: str
    report_year: int
    scope1_tco2: Optional[float] = None
    scope2_tco2: Optional[float] = None
    production_mt: Optional[float] = None
    net_zero_year: Optional[int] = None
    confidence: float


def parse_response(raw: str):
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return None, "No JSON object found via regex"
    try:
        record = SteelDisclosure.model_validate_json(match.group(0))
        return record.model_dump(), None
    except ValidationError as e:
        return None, f"ValidationError: {e}"
    except Exception as e:
        return None, f"ParseError: {e}"


parsed_rows = []
failures = []

for idx, row in recorded.iterrows():
    rec, err = parse_response(row["raw_text"])
    if rec is not None:
        parsed_rows.append({"id": row["id"], "kind": row["kind"], "sample": row.get("sample"), **rec})
    else:
        failures.append({"row_idx": idx, "id": row["id"], "kind": row["kind"], "sample": row.get("sample"), "raw_text": row["raw_text"], "error": err})

pred = pd.DataFrame(parsed_rows).astype({"report_year": "Int64", "net_zero_year": "Int64"})
print(f"Total recorded: {len(recorded)} | Parsed successfully: {len(pred)} | Failures: {len(failures)}")


Total recorded: 120 | Parsed successfully: 119 | Failures: 1


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [5]:
# Inspect parsing failures and text wrapping
print(f"Total parsing failures: {len(failures)}")
for f in failures:
    print(f"- ID: {f['id']} | Kind: {f['kind']} | Sample: {f['sample']}")
    print(f"  Error: {f['error']}")
    print(f"  Raw text preview: {repr(f['raw_text'][:160])}\n")

wrapped_count = sum(not (r.strip().startswith("{") and r.strip().endswith("}")) for r in recorded["raw_text"])
print(f"Responses wrapped in text / markdown fences: {wrapped_count} / {len(recorded)} ({wrapped_count / len(recorded):.1%})")

# Plausibility check: report parsed-but-implausible values separately
implausible = pred[(pred["net_zero_year"] < 2020) | (pred["net_zero_year"] > 2100)]
print(f"\nImplausible net_zero_year values ({len(implausible)} sample answers, kept in pred):")
display(implausible[["id", "kind", "sample", "net_zero_year", "confidence"]])


Total parsing failures: 1
- ID: E11 | Kind: greedy | Sample: 0
  Error: ValidationError: 1 validation error for SteelDisclosure
production_mt
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='8.1 Mt', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/float_parsing
  Raw text preview: '{\n  "company": "Anatolia Celik",\n  "report_year": 2024,\n  "scope1_tco2": 4120000,\n  "scope2_tco2": 2350000,\n  "production_mt": "8.1 Mt",\n  "net_zero_year": 2053'

Responses wrapped in text / markdown fences: 7 / 120 (5.8%)

Implausible net_zero_year values (5 sample answers, kept in pred):


,id,kind,sample,net_zero_year,confidence
38,E07,sample,1,2214,0.910
39,E07,sample,2,2214,0.880
66,E12,sample,0,20500,0.920
96,E17,sample,0,205,0.880
104,E18,sample,2,20500,0.910


**Your answer (Q2):**

Only 1 of the 120 recorded responses (0.8%) fails to parse: E11 greedy, where the model output "production_mt": "8.1 Mt" with the unit string attached, triggering a Pydantic validation error; 7 of the 120 responses (5.8%) required our regex because the JSON was enclosed in markdown code fences or conversational text. In production, one option is automated re-prompting with the error message (risk: added latency and cost, plus the risk that the model hallucinates a valid number when data is absent), and a second option is routing failed parses to human review (risk: analyst queues and linear labor cost). Counting a parse failure as wrong across all six fields is the honest benchmark choice because an unparseable response delivers zero usable data to downstream databases; scoring only parsed fields would reward models with survivorship bias.


> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [6]:
# Optional A2: Compare recorded greedy predictions with gold on E03, E09, E19
if BACKEND != "recorded" and MODEL is not None:
    print(f"Running live model: {MODEL} on {BACKEND} backend...")
    live_results = []
    for eid in ["E03", "E09", "E19"]:
        raw_out = ask_llm(PROMPT.format(text=excerpts.loc[excerpts["id"] == eid, "text"].values[0]), temperature=0.0)
        rec, err = parse_response(raw_out)
        live_results.append({"id": eid, "parsed": rec, "error": err})
    display(pd.DataFrame(live_results))
else:
    print("BACKEND is 'recorded' (no live LLM configured). Comparing recorded greedy predictions with gold for E03, E09, E19:")
    comp_df = pred[(pred["kind"] == "greedy") & (pred["id"].isin(["E03", "E09", "E19"]))].set_index("id")[FIELDS]
    gold_sub = gold.loc[["E03", "E09", "E19"], FIELDS]
    display(pd.concat({"Pred (Greedy)": comp_df, "Gold": gold_sub}, axis=1))


BACKEND is 'recorded' (no live LLM configured). Comparing recorded greedy predictions with gold for E03, E09, E19:


Pred (Greedy)                                        \
                       company report_year   scope1_tco2 scope2_tco2   
id                                                                     
E03  Lianhe Iron & Steel Group        2023 4,108,000.000 512,000.000   
E09              Ostend Metals        2024 8,700,000.000         NaN   
E19             Bluewater Iron        2024 5,100,000.000 410,000.000   

                                                      Gold              \
    production_mt net_zero_year                    company report_year   
id                                                                       
E03        21.400          2060  Lianhe Iron & Steel Group        2023   
E09         4.400          2050              Ostend Metals        2024   
E19         3.300          2050             Bluewater Iron        2024   

                                                              
       scope1_tco2   scope2_tco2 production_mt net_zero_year  
id                                                            
E03 41,080,000.000 5,120,000.000        21.400          2060  
E09            NaN           NaN         4.400          2050  
E19  5,100,000.000   410,000.000         2.990          2050

---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [7]:
def field_correct(field: str, pred_val, gold_val) -> bool:
    p_na, g_na = pd.isna(pred_val), pd.isna(gold_val)
    if p_na and g_na:
        return True
    if p_na != g_na:
        return False
    if field == "company":
        p = re.sub(r"[^\w\s]", "", str(pred_val)).strip().lower()
        g = re.sub(r"[^\w\s]", "", str(gold_val)).strip().lower()
        return (p in g) or (g in p)
    if field in ["report_year", "net_zero_year"]:
        return int(pred_val) == int(gold_val)
    if field in ["scope1_tco2", "scope2_tco2", "production_mt"]:
        p, g = float(pred_val), float(gold_val)
        if g == 0:
            return abs(p - g) < 1e-6
        return abs(p - g) / abs(g) <= 0.02
    return False


def score(df: pd.DataFrame) -> pd.DataFrame:
    df_indexed = df.set_index("id")
    rows = []
    for eid in gold.index:
        has_pred = eid in df_indexed.index
        for f in FIELDS:
            g_val = gold.loc[eid, f]
            p_val = df_indexed.loc[eid, f] if has_pred else None
            corr = field_correct(f, p_val, g_val) if has_pred else False
            rows.append({"id": eid, "field": f, "pred": p_val, "gold": g_val, "correct": corr})
    return pd.DataFrame(rows)


greedy_pred = pred[pred["kind"] == "greedy"].copy()
greedy_scores = score(greedy_pred)

field_accuracy = (greedy_scores.groupby("field")["correct"]
                  .agg(correct="sum", total="count", accuracy="mean")
                  .sort_values(by="accuracy", ascending=True))
display(field_accuracy.style.format({"accuracy": "{:.1%}"}))

overall_accuracy = greedy_scores["correct"].mean()
print(f"\nOverall Greedy Accuracy: {overall_accuracy:.1%} ({greedy_scores['correct'].sum()}/{len(greedy_scores)} fields)")


,correct,total,accuracy
field,,,
scope1_tco2,15,20,75.0%
net_zero_year,17,20,85.0%
scope2_tco2,17,20,85.0%
production_mt,18,20,90.0%
report_year,18,20,90.0%
company,19,20,95.0%



Overall Greedy Accuracy: 86.7% (104/120 fields)


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [8]:
# Detail wrong predictions and categorize errors
wrong_greedy = greedy_scores[~greedy_scores["correct"]].copy()
print(f"Total wrong fields: {len(wrong_greedy)} / {len(greedy_scores)}")
display(wrong_greedy[["id", "field", "pred", "gold"]])

def classify_error(row):
    eid, f = row["id"], row["field"]
    if eid == "E11":
        return "Parse validation failure"
    if eid in ["E03", "E19"]:
        return "Unit conversion"
    if eid in ["E10", "E20"]:
        return "Wrong date as net zero"
    if eid in ["E07", "E08"]:
        return "Wrong period"
    if eid == "E05":
        return "Computed value not reported"
    if eid == "E09":
        return "Combined scope as Scope 1"
    if eid == "E18":
        return "Non-disclosure as zero"
    return "Other"

wrong_greedy["error_type"] = wrong_greedy.apply(classify_error, axis=1)
taxonomy = (wrong_greedy.groupby("error_type")
            .agg(Fields=("field", "count"), Excerpts=("id", "nunique"))
            .reset_index()
            .rename(columns={"error_type": "Error Type"}))
taxonomy["Share of Errors"] = taxonomy["Fields"] / len(wrong_greedy)
taxonomy = taxonomy.sort_values(by=["Fields", "Excerpts"], ascending=False)
display(taxonomy.style.format({"Share of Errors": "{:.1%}"}))


Total wrong fields: 16 / 120


,id,field,pred,gold
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000"
15,E03,scope2_tco2,"512,000.000","5,120,000.000"
26,E05,scope1_tco2,"42,042,000.000",NaN
38,E07,scope1_tco2,"14,200,000.000","13,800,000.000"
43,E08,report_year,2025,2024
50,E09,scope1_tco2,"8,700,000.000",NaN
59,E10,net_zero_year,2030,<NA>
60,E11,company,None,Anatolia Celik
61,E11,report_year,None,2024
62,E11,scope1_tco2,None,"4,120,000.000"


,Error Type,Fields,Excerpts,Share of Errors
3,Parse validation failure,6,1,37.5%
4,Unit conversion,3,2,18.8%
5,Wrong date as net zero,2,2,12.5%
6,Wrong period,2,2,12.5%
0,Combined scope as Scope 1,1,1,6.2%
1,Computed value not reported,1,1,6.2%
2,Non-disclosure as zero,1,1,6.2%


**Your answer (Q3):**

Overall greedy accuracy is 86.7% (104/120 fields), with Scope 1 emissions being the worst field at 75.0% (15/20). Across the 16 errors, the single Pydantic parse failure on E11 accounts for the most failed fields (6 fields), but across excerpts, unit conversion errors (E03 10,000-tonne multiplier and E19 short tons), wrong dates for net zero (E10 interim target and E20 commissioning date), and wrong time periods (E07 prior year and E08 fiscal end-year) are the most frequent failure modes (2 excerpts each). In Q1, human annotators in our group easily made the unit errors (overlooking short tons or the 10,000 multiplier) and fiscal-year confusion, but a human would not cause a schema validation crash like E11.


## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [9]:
k = int(greedy_scores["correct"].sum())
n = int(len(greedy_scores))

# 1. Bayesian Beta-Binomial: Beta(1 + k, 1 + n - k)
beta_ci = stats.beta(1 + k, 1 + n - k).ppf([0.05, 0.95])
p_gt_95_beta = 1.0 - stats.beta(1 + k, 1 + n - k).cdf(0.95)

# 2. Bootstrap over excerpts (resample 20 excerpts with replacement, 2,000 times)
rng = np.random.default_rng(SEED)
excerpt_summary = greedy_scores.groupby("id")["correct"].agg(["sum", "count"]).reset_index()
N_excerpts = len(excerpt_summary)

boot_accs = []
for _ in range(2000):
    sample_indices = rng.integers(0, N_excerpts, size=N_excerpts)
    sampled = excerpt_summary.iloc[sample_indices]
    boot_accs.append(sampled["sum"].sum() / sampled["count"].sum())

boot_ci = np.percentile(boot_accs, [5, 95])

print(f"k = {k}, n = {n}")
print(f"Beta(1, 1) 90% Credible Interval: [{beta_ci[0]:.3f}, {beta_ci[1]:.3f}] ({beta_ci[0]:.1%}, {beta_ci[1]:.1%}) | Width: {beta_ci[1] - beta_ci[0]:.1%}")
print(f"Bootstrap 90% Confidence Interval: [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}] ({boot_ci[0]:.1%}, {boot_ci[1]:.1%}) | Width: {boot_ci[1] - boot_ci[0]:.1%}")
print(f"P(accuracy > 95% | data) [Beta]:   {p_gt_95_beta:.4f} ({p_gt_95_beta:.2%})")


k = 104, n = 120
Beta(1, 1) 90% Credible Interval: [0.806, 0.908] (80.6%, 90.8%) | Width: 10.2%
Bootstrap 90% Confidence Interval: [0.775, 0.933] (77.5%, 93.3%) | Width: 15.8%
P(accuracy > 95% | data) [Beta]:   0.0001 (0.01%)


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

The Beta(1,1) 90% credible interval is [80.6%, 90.8%], while the 2,000-sample bootstrap interval over excerpts is wider at [77.5%, 93.3%]. The bootstrap interval is wider because the six fields of an excerpt are not independent draws—when an excerpt fails schema validation (E11) or has an ambiguous layout, errors cluster, so our effective sample size is closer to 20 excerpts than 120 independent fields. Halving the interval width requires four times as much data, or roughly 80 labeled excerpts. A vendor's claim of 95% accuracy is inconsistent with this greedy model, as the probability of true accuracy exceeding 95% under our posterior is only 0.01%.


## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [10]:
downstream = pd.DataFrame(index=gold.index)
greedy_idx = greedy_pred.set_index("id")

downstream["scope1_llm"] = greedy_idx["scope1_tco2"]
downstream["scope1_gold"] = gold["scope1_tco2"]
downstream["prod_llm"] = greedy_idx["production_mt"]
downstream["prod_gold"] = gold["production_mt"]

downstream["scope1_llm_filled"] = downstream["scope1_llm"].fillna(0)
downstream["scope1_gold_filled"] = downstream["scope1_gold"].fillna(0)
downstream["error_t"] = downstream["scope1_llm_filled"] - downstream["scope1_gold_filled"]

total_llm_mt = downstream["scope1_llm_filled"].sum() / 1e6
total_gold_mt = downstream["scope1_gold_filled"].sum() / 1e6
net_error_mt = downstream["error_t"].sum() / 1e6
abs_error_mt = downstream["error_t"].abs().sum() / 1e6

int_llm = downstream["scope1_llm"] / (downstream["prod_llm"] * 1e6)
int_gold = downstream["scope1_gold"] / (downstream["prod_gold"] * 1e6)
valid_llm = int_llm.dropna()
valid_gold = int_gold.dropna()
common_idx = valid_llm.index.intersection(valid_gold.index)

print(f"Scope 1 Total (LLM):  {total_llm_mt:,.3f} MtCO2e")
print(f"Scope 1 Total (Gold): {total_gold_mt:,.3f} MtCO2e")
print(f"Net Error:            {net_error_mt:+,.3f} MtCO2e ({net_error_mt / total_gold_mt:+.1%})")
print(f"Absolute Error:       {abs_error_mt:,.3f} MtCO2e ({abs_error_mt / total_gold_mt:.1%})")
print(f"Median Intensity (LLM, n={len(valid_llm)}):   {valid_llm.median():.3f} tCO2 / t steel")
print(f"Median Intensity (Gold, n={len(valid_gold)}):  {valid_gold.median():.3f} tCO2 / t steel")
print(f"Median Intensity (Common, n={len(common_idx)}): LLM={int_llm.loc[common_idx].median():.3f} vs Gold={int_gold.loc[common_idx].median():.3f} tCO2 / t steel")

print("\nTop contributing excerpts to Scope 1 error:")
display(downstream[["scope1_llm", "scope1_gold", "error_t"]]
        .sort_values(by="error_t", key=abs, ascending=False).head(5))


Scope 1 Total (LLM):  252.112 MtCO2e
Scope 1 Total (Gold): 242.062 MtCO2e
Net Error:            +10.050 MtCO2e (+4.2%)
Absolute Error:       92.234 MtCO2e (38.1%)
Median Intensity (LLM, n=18):   1.895 tCO2 / t steel
Median Intensity (Gold, n=17):  1.865 tCO2 / t steel
Median Intensity (Common, n=16): LLM=1.845 vs Gold=1.878 tCO2 / t steel

Top contributing excerpts to Scope 1 error:


,scope1_llm,scope1_gold,error_t
id,,,
E05,"42,042,000.000",NaN,"42,042,000.000"
E03,"4,108,000.000","41,080,000.000","-36,972,000.000"
E09,"8,700,000.000",NaN,"8,700,000.000"
E11,NaN,"4,120,000.000","-4,120,000.000"
E07,"14,200,000.000","13,800,000.000","400,000.000"


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

On the sector total, the LLM estimates 252.112 MtCO2e against a gold total of 242.062 MtCO2e, giving a net error of +10.050 MtCO2e (+4.2%) and a gross absolute error of 92.234 MtCO2e (38.1%). E05 (+42.042 Mt, where the model computed Scope 1 from intensity and production) and E03 (-36.972 Mt, where it missed the 10,000-tonne unit) account for 86% of the absolute error. The median intensity barely changes (1.895 vs 1.865 tCO2/t steel across available companies, or 1.845 vs 1.878 on the 16 common ones) because the median is an order statistic: large errors in the tails only shift the middle rank by one position, whereas linear portfolio totals absorb the full arithmetic error. Alongside the total, an analyst must report the 92.2 Mt absolute error sum, the data coverage, and recommend manual verification of the largest emitters.


---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [11]:
sample_preds = pred[pred["kind"] == "sample"].copy()

votes_records = []
for eid in gold.index:
    eid_samples = sample_preds[sample_preds["id"] == eid]
    n_samples = len(eid_samples)
    for f in FIELDS:
        g_val = gold.loc[eid, f]
        if n_samples == 0:
            votes_records.append({"id": eid, "field": f, "vote": None, "agreement": 0.0, "correct": False})
            continue

        def to_key(val):
            if pd.isna(val):
                return None
            if hasattr(val, "item"):
                return val.item()
            return val

        raw_vals = eid_samples[f].values
        str_keys = [json.dumps(to_key(v)) for v in raw_vals]
        counts = pd.Series(str_keys).value_counts()
        winner_key = counts.index[0]
        winner_count = counts.iloc[0]
        winner_val = json.loads(winner_key)
        agreement = winner_count / 5.0
        corr = field_correct(f, winner_val, g_val)
        votes_records.append({"id": eid, "field": f, "vote": winner_val, "agreement": agreement, "correct": corr})

votes = pd.DataFrame(votes_records)

def assign_bin(agr):
    if agr >= 1.0:
        return "5/5"
    if agr >= 0.8:
        return "4/5"
    return "<=3/5"

votes["agreement_bin"] = votes["agreement"].apply(assign_bin)

vote_accuracy = votes["correct"].mean()
print(f"Greedy Accuracy: {overall_accuracy:.1%} | Majority Vote Accuracy: {vote_accuracy:.1%}")

bin_table = (votes.groupby("agreement_bin")["correct"]
             .agg(total="count", correct="sum", accuracy="mean")
             .loc[["5/5", "4/5", "<=3/5"]])
display(bin_table.style.format({"accuracy": "{:.1%}"}))


Greedy Accuracy: 86.7% | Majority Vote Accuracy: 95.0%


,total,correct,accuracy
agreement_bin,,,
5/5,93,91,97.8%
4/5,15,14,93.3%
<=3/5,12,9,75.0%


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [12]:
k_5 = int(bin_table.loc["5/5", "correct"])
n_5 = int(bin_table.loc["5/5", "total"])
beta_5_ci = stats.beta(1 + k_5, 1 + n_5 - k_5).ppf([0.05, 0.95])
p_gt_95_vote5 = 1.0 - stats.beta(1 + k_5, 1 + n_5 - k_5).cdf(0.95)

k_vote = int(votes["correct"].sum())
n_vote = len(votes)
vote_beta_ci = stats.beta(1 + k_vote, 1 + n_vote - k_vote).ppf([0.05, 0.95])

greedy_no_e11 = greedy_scores[greedy_scores["id"] != "E11"]
acc_no_e11 = greedy_no_e11["correct"].mean()

print(f"5/5 agreement bin: {k_5}/{n_5} correct ({k_5/n_5:.1%})")
print(f"Beta(1, 1) 90% Credible Interval for 5/5: [{beta_5_ci[0]:.3f}, {beta_5_ci[1]:.3f}] ({beta_5_ci[0]:.1%}, {beta_5_ci[1]:.1%})")
print(f"P(accuracy > 95% | 5/5 bin) [Beta]:        {p_gt_95_vote5:.4f} ({p_gt_95_vote5:.1%})")
print(f"Overall Vote Beta(1, 1) 90% Credible Interval: [{vote_beta_ci[0]:.3f}, {vote_beta_ci[1]:.3f}] ({vote_beta_ci[0]:.1%}, {vote_beta_ci[1]:.1%})")
print(f"Greedy accuracy without E11: {greedy_no_e11['correct'].sum()}/{len(greedy_no_e11)} = {acc_no_e11:.1%}")

# Display the 6 errors that survive majority voting
vote_errors = votes[~votes["correct"]].copy()
print(f"\nThe 6 errors surviving majority voting (out of 120 fields):")
display(vote_errors[["id", "field", "vote", "agreement"]])


5/5 agreement bin: 91/93 correct (97.8%)
Beta(1, 1) 90% Credible Interval for 5/5: [0.935, 0.991] (93.5%, 99.1%)
P(accuracy > 95% | 5/5 bin) [Beta]:        0.8546 (85.5%)
Overall Vote Beta(1, 1) 90% Credible Interval: [0.904, 0.973] (90.4%, 97.3%)
Greedy accuracy without E11: 104/114 = 91.2%

The 6 errors surviving majority voting (out of 120 fields):


,id,field,vote,agreement
14,E03,scope1_tco2,"4,108,000.000",1.000
50,E09,scope1_tco2,"8,700,000.000",0.600
59,E10,net_zero_year,2030,0.600
64,E11,production_mt,8.100,0.600
65,E11,net_zero_year,2050,0.800
112,E19,production_mt,3.300,1.000


**Your answer (Q6):**

Majority voting lifts overall accuracy from 86.7% (or 91.2% excluding the E11 parse failure) to 95.0% (114/120 fields), with accuracy grading cleanly by agreement: 97.8% (91/93) at 5/5, 93.3% (14/15) at 4/5, and 75.0% (9/12) at 3/5 or less. Low agreement is a strong warning signal: error rates jump from 2.2% when unanimous to 25.0% when split. For the 5/5 agreement bin, the Beta(1,1) 90% credible interval is [93.5%, 99.1%]; because the lower bound is 93.5%, we cannot strictly claim accuracy is above 95% at 90% confidence, even though the point estimate is 97.8%.


## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [13]:
# Identify confidently wrong fields: 5/5 agreement but incorrect
confidently_wrong = votes[(votes["agreement"] == 1.0) & (~votes["correct"])].copy()
confidently_wrong["gold"] = confidently_wrong.apply(lambda r: gold.loc[r["id"], r["field"]], axis=1)
print(f"Confidently wrong fields (agreement = 5/5, correct = False): {len(confidently_wrong)}")
display(confidently_wrong[["id", "field", "vote", "gold"]])


Confidently wrong fields (agreement = 5/5, correct = False): 2


,id,field,vote,gold
14,E03,scope1_tco2,"4,108,000.000","41,080,000.000"
112,E19,production_mt,3.300,2.990


**Your answer (Q7):**

Two fields are wrong despite unanimous 5/5 agreement: E03 Scope 1 (4,108,000 vs 41,080,000) and E19 production (3.3 vs 2.99 Mt). Both are unit conversion blind spots where the model extracts the printed number correctly but consistently overlooks the unit definition ("10,000 tonnes" and "short tons"). Resampling cannot catch these because they are systematic model biases rather than random sampling variance; drawing more samples from the same prompt and model simply repeats the same misinterpretation with high agreement. Fixing them requires a regex rule for unusual units paired with human review, or prompt examples with non-SI units, analogous to the common-mode sensor or emission-factor bias discussed in Session 4.


## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [14]:
sample_eval = []
for (eid, s_idx), grp in sample_preds.groupby(["id", "sample"]):
    all_correct = True
    for f in FIELDS:
        pval = grp[f].values[0]
        gval = gold.loc[eid, f]
        if not field_correct(f, pval, gval):
            all_correct = False
            break
    conf = grp["confidence"].values[0]
    sample_eval.append({"id": eid, "sample": s_idx, "confidence": conf, "all_correct": all_correct})

calib_df = pd.DataFrame(sample_eval)
print(f"Mean Stated Confidence:            {calib_df['confidence'].mean():.1%}")
print(f"Share with All 6 Fields Correct:   {calib_df['all_correct'].mean():.1%} ({calib_df['all_correct'].sum()}/{len(calib_df)})")

calib_df["conf_bin"] = pd.cut(calib_df["confidence"], bins=[0.0, 0.90, 0.93, 0.96, 1.0])
calib_table = (calib_df.groupby("conf_bin", observed=False)["all_correct"]
               .agg(total="count", fully_correct="sum", accuracy="mean"))
display(calib_table.style.format({"accuracy": "{:.1%}"}))

for eid in ["E03", "E09", "E19"]:
    e_rows = calib_df[calib_df["id"] == eid]
    print(f"Excerpt {eid} stated confidences: {e_rows['confidence'].tolist()} ({int(e_rows['all_correct'].sum())}/5 fully correct)")


Mean Stated Confidence:            92.8%
Share with All 6 Fields Correct:   55.0% (55/100)


,total,fully_correct,accuracy
conf_bin,,,
"(0.0, 0.9]",21,6,28.6%
"(0.9, 0.93]",35,16,45.7%
"(0.93, 0.96]",38,28,73.7%
"(0.96, 1.0]",6,5,83.3%


Excerpt E03 stated confidences: [0.89, 0.89, 0.91, 0.93, 0.94] (0/5 fully correct)
Excerpt E09 stated confidences: [0.92, 0.91, 0.93, 0.95, 0.9] (0/5 fully correct)
Excerpt E19 stated confidences: [0.91, 0.94, 0.85, 0.93, 0.95] (0/5 fully correct)


**Your answer (Q8):**

The model is severely overconfident: across sample runs its mean stated confidence is 92.8%, but only 55.0% (55/100) of responses are fully correct on all six fields, and it assigned 0.85 to 0.95 confidence even on excerpts where every sample had an error (E03, E09, E19). However, confidence is still monotonically useful for ranking: the share of fully correct responses rises from 28.6% for confidence under 0.90, to 45.7% for 0.90–0.93, 73.7% for 0.93–0.96, and 83.3% above 0.96 (though n=6 in the top bin). Confidence can therefore be used to prioritize human review queues, but never treated as an objective probability of correctness.


## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [15]:
COST_REVIEW = 4
COST_ERROR = 150

unit_pattern = re.compile(r"short tons?|10,000 tonnes", re.IGNORECASE)
unit_flagged_eids = set(excerpts[excerpts["text"].str.contains(unit_pattern)]["id"])

policy_conditions = [
    ("P0: Trust vote everywhere", lambda r: False),
    ("P1: Review agreement < 5/5", lambda r: r["agreement"] < 1.0),
    ("P2: Review agreement <= 3/5", lambda r: r["agreement"] <= 0.6),
    ("P3: P1 + Unit trigger (regex)", lambda r: (r["agreement"] < 1.0) or (r["id"] in unit_flagged_eids))
]

policy_results = []
for name, condition in policy_conditions:
    needs_review = votes.apply(condition, axis=1)
    n_reviewed = int(needs_review.sum())
    surviving_errors = int((~needs_review & ~votes["correct"]).sum())
    cost_rev = n_reviewed * COST_REVIEW
    cost_err = surviving_errors * COST_ERROR
    total_cost = cost_rev + cost_err
    policy_results.append({
        "Policy": name,
        "Reviewed Fields": n_reviewed,
        "Residual Errors": surviving_errors,
        "Review Cost (EUR)": cost_rev,
        "Error Cost (EUR)": cost_err,
        "Total Cost (EUR)": total_cost
    })

policy_df = pd.DataFrame(policy_results).set_index("Policy")
display(policy_df)

break_even_rate = COST_REVIEW / COST_ERROR
print(f"\nBreak-even error rate: EUR {COST_REVIEW} / EUR {COST_ERROR} = {break_even_rate:.1%}")
for b in ["5/5", "4/5", "<=3/5"]:
    b_err = 1.0 - bin_table.loc[b, "accuracy"]
    print(f"Error rate in bin {b:>5}: {b_err:.1%} (vs break-even {break_even_rate:.1%})")


,Reviewed Fields,Residual Errors,Review Cost (EUR),Error Cost (EUR),Total Cost (EUR)
Policy,,,,,
P0: Trust vote everywhere,0,6,0,900,900
P1: Review agreement < 5/5,27,2,108,300,408
P2: Review agreement <= 3/5,12,3,48,450,498
P3: P1 + Unit trigger (regex),37,0,148,0,148



Break-even error rate: EUR 4 / EUR 150 = 2.7%
Error rate in bin   5/5: 2.2% (vs break-even 2.7%)
Error rate in bin   4/5: 6.7% (vs break-even 2.7%)
Error rate in bin <=3/5: 25.0% (vs break-even 2.7%)


**Your answer (Q9):**

Policy P1 (reviewing all 27 fields with agreement < 5/5) is cheapest at EUR 408 (EUR 108 review + EUR 300 residual error), easily beating P0 (EUR 900) and P2 (EUR 498). P1 beats P2 because the break-even error rate is EUR 4 / EUR 150 = 2.7%, and the 4/5 agreement bin has an error rate of 6.7%, meaning paying EUR 4 to review those fields prevents an expected EUR 10 error cost. P1 leaves only the two unanimous unit errors (E03 Scope 1 and E19 production); adding a simple regex trigger for unit keywords ("short tons", "10,000 tonnes") catches both, cutting total cost to EUR 148, though this rule is tuned in-sample. Note that all three policies assume five sample calls per excerpt, which costs five times more in LLM API tokens than a single greedy run.


---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = corpus["chunks"]
queries = corpus["queries"]

vectorizer = TfidfVectorizer(stop_words="english")
X_chunks = vectorizer.fit_transform(chunks)
X_queries = vectorizer.transform([q["query"] for q in queries])
tfidf_sims = cosine_similarity(X_queries, X_chunks)

retrieval_records = []
for i, q in enumerate(queries):
    top3 = np.argsort(-tfidf_sims[i])[:3].tolist()
    rel = q["relevant"][0]
    h1 = (top3[0] == rel)
    h3 = (rel in top3)
    retrieval_records.append({
        "qid": q["qid"],
        "query": q["query"],
        "relevant_chunk": rel,
        "top3_chunks": top3,
        "hit@1": h1,
        "hit@3": h3
    })

retrieval = pd.DataFrame(retrieval_records)
print(f"TF-IDF hit@1: {retrieval['hit@1'].mean():.1%} ({retrieval['hit@1'].sum()}/{len(retrieval)})")
print(f"TF-IDF hit@3: {retrieval['hit@3'].mean():.1%} ({retrieval['hit@3'].sum()}/{len(retrieval)})")
display(retrieval)


TF-IDF hit@1: 66.7% (4/6)
TF-IDF hit@3: 66.7% (4/6)


,qid,query,relevant_chunk,top3_chunks,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,3,"[1, 5, 12]",False,False
1,Q2,net zero target year,5,"[5, 0, 13]",True,True
2,Q3,crude steel production 2024,2,"[2, 3, 0]",True,True
3,Q4,investment in hydrogen DRI capex,6,"[6, 0, 2]",True,True
4,Q5,impact of the carbon price on EBITDA,8,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,9,"[3, 6, 1]",False,False


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [17]:
failed_queries = retrieval[~retrieval["hit@1"]]
analyzer = vectorizer.build_analyzer()

for _, r in failed_queries.iterrows():
    qid = r["qid"]
    rel_idx = r["relevant_chunk"]
    ret_idx = r["top3_chunks"][0]
    q_tokens = analyzer(r["query"])
    rel_tokens = analyzer(chunks[rel_idx])
    ret_tokens = analyzer(chunks[ret_idx])
    q_idx = int(qid[1:]) - 1
    all_ranks = np.argsort(-tfidf_sims[q_idx]).tolist()
    rel_rank = all_ranks.index(rel_idx) + 1
    
    print(f"================ Query {qid} ================")
    print(f"Query: \"{r['query']}\"")
    print(f"Tokens extracted by TF-IDF: {q_tokens}")
    print(f"[Rank 1 Retrieved Chunk #{ret_idx}] (sim={tfidf_sims[q_idx, ret_idx]:.3f}):")
    print(f"  Text: {chunks[ret_idx].strip()[:140]}...")
    print(f"  Shared tokens with query: {set(q_tokens).intersection(ret_tokens)}")
    print(f"[Ground-Truth Chunk #{rel_idx}] (Rank #{rel_rank}, sim={tfidf_sims[q_idx, rel_idx]:.3f}):")
    print(f"  Text: {chunks[rel_idx].strip()[:140]}...")
    print(f"  Shared tokens with query: {set(q_tokens).intersection(rel_tokens)}")
    print("-" * 60 + "\n")


================ Query Q1 ================
Query: "What were Scope 1 emissions in 2024?"
Tokens extracted by TF-IDF: ['scope', 'emissions', '2024']
[Rank 1 Retrieved Chunk #1] (sim=0.338):
  Text: About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 cover...
  Shared tokens with query: {'emissions', 'scope'}
[Ground-Truth Chunk #3] (Rank #4, sim=0.184):
  Text: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an...
  Shared tokens with query: {'2024', 'emissions'}
------------------------------------------------------------

================ Query Q6 ================
Query: "flooding exposure of our plants"
Tokens extracted by TF-IDF: ['flooding', 'exposure', 'plants']
[Rank 1 Retrieved Chunk #3] (sim=0.208):
  Text: Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter pl

**Your answer (Q10):**

TF-IDF achieves 66.7% (4/6) for both hit@1 and hit@3, missing Q1 and Q6 entirely. For Q1, the default tokenizer drops single-character words, turning "Scope 1" into "scope", so Chunk 1 (methodology) wins on term overlap while Chunk 3 (the data) ranks 4th because it uses "direct greenhouse gas emissions" without the word "scope"; for Q6, Chunk 9 uses "riverine inundation" and shares no words with the query, allowing Chunk 3 to win by accident via the word "plants". Dense embeddings resolve Q6 by matching "flooding" to "riverine inundation", but still fail Q1 because linking Scope 1 to direct emissions requires domain knowledge. In RAG pipelines, a retrieval miss is far more dangerous than an extraction error because it starves the model of context, creating a systematic failure that resampling and voting cannot detect.


> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [18]:
# Optional D2: Dense embeddings retrieval using SentenceTransformer
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from sentence_transformers import SentenceTransformer
    dense_model = SentenceTransformer("all-MiniLM-L6-v2")

    chunk_embeddings = dense_model.encode(chunks, show_progress_bar=False)
    query_embeddings = dense_model.encode([q["query"] for q in queries], show_progress_bar=False)
    dense_sims = cosine_similarity(query_embeddings, chunk_embeddings)

    dense_records = []
    for i, q in enumerate(queries):
        top3 = np.argsort(-dense_sims[i])[:3].tolist()
        rel = q["relevant"][0]
        h1 = (top3[0] == rel)
        h3 = (rel in top3)
        dense_records.append({
            "qid": q["qid"],
            "query": q["query"],
            "relevant_chunk": rel,
            "top3_chunks": top3,
            "hit@1": h1,
            "hit@3": h3
        })

    dense_retrieval = pd.DataFrame(dense_records)
    print(f"Dense Embeddings hit@1: {dense_retrieval['hit@1'].mean():.1%} ({dense_retrieval['hit@1'].sum()}/{len(dense_retrieval)})")
    print(f"Dense Embeddings hit@3: {dense_retrieval['hit@3'].mean():.1%} ({dense_retrieval['hit@3'].sum()}/{len(dense_retrieval)})")
    display(dense_retrieval)

    still_failed = dense_retrieval[~dense_retrieval["hit@1"]]
    for _, r in still_failed.iterrows():
        print(f"Query still failing: {r['qid']} (\"{r['query']}\") -> retrieved chunk {r['top3_chunks'][0]} instead of {r['relevant_chunk']}")

except Exception as e:
    print(f"Dense embeddings unavailable ({e}).")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense Embeddings hit@1: 83.3% (5/6)
Dense Embeddings hit@3: 83.3% (5/6)


,qid,query,relevant_chunk,top3_chunks,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,3,"[5, 12, 1]",False,False
1,Q2,net zero target year,5,"[5, 0, 6]",True,True
2,Q3,crude steel production 2024,2,"[2, 0, 3]",True,True
3,Q4,investment in hydrogen DRI capex,6,"[6, 8, 7]",True,True
4,Q5,impact of the carbon price on EBITDA,8,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,9,"[9, 10, 3]",True,True


Query still failing: Q1 ("What were Scope 1 emissions in 2024?") -> retrieved chunk 5 instead of 3


---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

We evaluated an LLM pipeline on 20 steel disclosures. Single-run accuracy is 87% (90% credible interval: 81–91%), rising to 95% (90–97%) with five-sample voting. Its main failure mode is misconverting non-standard units (like short tons or 10,000-tonne multipliers), where all five samples agree on the wrong number.

We recommend adopting five-sample voting with human review whenever samples disagree (EUR 408 per 20 reports), plus a unit-keyword check (reducing cost to EUR 148), which is very likely to prevent material errors. However, our 20-report sample is small, leaving out-of-sample generalization unverified.
